# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [3]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = Path(r"C:\dev\llm-data-analysis-course")

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\Users\82102\Desktop\Ozzang\03 대학원\llm-data-analysis-study\chapter06
프로젝트 루트: C:\dev\llm-data-analysis-course
전처리 데이터 폴더: C:\dev\llm-data-analysis-course\data\processed
보고서 폴더: C:\dev\llm-data-analysis-course\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [4]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [5]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [6]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 0
주문 시작일: 2025-09-16 00:00:00
주문 종료일: 2026-09-15 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [7]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [8]:
customers['age'].describe()


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [9]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


In [10]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,F,84,56.0
1,M,66,44.0


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [11]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


In [12]:
products['price'].describe()


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [13]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [14]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


In [15]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


In [16]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-09-16 00:00:00
주문 종료일: 2026-09-15 00:00:00
주문 월 개수: 13


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [17]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [18]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (764, 6)
병합 후 sales_items: (764, 9)
상품명 누락: 0
카테고리 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,100,3,102000,306000,도서 상품 100,도서,102000
1,2,1,87,5,25000,125000,도서 상품 087,도서,25000
2,3,1,7,3,142000,426000,도서 상품 007,도서,142000
3,4,1,9,3,193000,579000,스포츠 상품 009,스포츠,193000
4,5,2,72,4,189000,756000,뷰티 상품 072,뷰티,189000


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [19]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14
0,도서,238,24645000,9.64
6,패션,220,23801000,9.31


In [20]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,468,50174000,19.63,111579.0
1,뷰티,376,47551000,18.60,117688.0
2,전자기기,401,41003000,16.04,101588.0
3,생활용품,390,34839000,13.63,96438.0
4,식품,240,33597000,13.14,137143.0
5,도서,238,24645000,9.64,106857.0
6,패션,220,23801000,9.31,115909.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [21]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (764, 12)
order_date 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,100,3,102000,306000,123,2026-07-15,card,completed,2026-07,Wednesday
1,2,1,87,5,25000,125000,123,2026-07-15,card,completed,2026-07,Wednesday
2,3,1,7,3,142000,426000,123,2026-07-15,card,completed,2026-07,Wednesday
3,4,1,9,3,193000,579000,123,2026-07-15,card,completed,2026-07,Wednesday
4,5,2,72,4,189000,756000,77,2025-09-30,naver_pay,cancelled,2025-09,Tuesday


In [22]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-09,4950000,9,550000.0
1,2025-10,25481000,27,943741.0
2,2025-11,18572000,22,844182.0
3,2025-12,28360000,31,914839.0
4,2026-01,30862000,31,995548.0
5,2026-02,14289000,17,840529.0
6,2026-03,25161000,35,718886.0
7,2026-04,21774000,25,870960.0
8,2026-05,26355000,29,908793.0
9,2026-06,18806000,25,752240.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [23]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [24]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [25]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('C:/dev/llm-data-analysis-course/reports/ch06_category_sales.csv'),
 WindowsPath('C:/dev/llm-data-analysis-course/reports/ch06_customer_city.csv'),
 WindowsPath('C:/dev/llm-data-analysis-course/reports/ch06_customer_sales.csv'),
 WindowsPath('C:/dev/llm-data-analysis-course/reports/ch06_eda_questions.csv'),
 WindowsPath('C:/dev/llm-data-analysis-course/reports/ch06_monthly_sales.csv'),
 WindowsPath('C:/dev/llm-data-analysis-course/reports/ch06_product_category.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [26]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: C:\dev\llm-data-analysis-course\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [27]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [ ]:
module_results['category_sales'].head()

,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [29]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.

city_sales = (
    customer_sales
    .groupby('city', as_index=False)
    .agg(total_sales=('total_sales', 'sum'))
)

city_comparison = customer_city.merge(
    city_sales,
    on='city',
    how='left'
)

city_comparison

,city,customer_count,customer_ratio,total_sales
0,성남,21,14.00,42030000
1,광주,17,11.33,22873000
2,부산,16,10.67,21276000
3,대구,15,10.00,22964000
4,서울,15,10.00,31980000
5,울산,14,9.33,20997000
6,인천,14,9.33,22030000
7,대전,14,9.33,20343000
8,수원,13,8.67,28751000
9,고양,11,7.33,22366000


In [33]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
category_comparison = product_category.merge(
    category_sales,
    on='category',
    how='left'
)

category_comparison

,category,product_count,product_ratio,total_quantity,total_sales,sales_ratio
0,스포츠,19,19.0,468,50174000,19.63
1,전자기기,17,17.0,401,41003000,16.04
2,생활용품,16,16.0,390,34839000,13.63
3,뷰티,16,16.0,376,47551000,18.60
4,도서,14,14.0,238,24645000,9.64
5,패션,11,11.0,220,23801000,9.31
6,식품,7,7.0,240,33597000,13.14


In [34]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
# 결제수단, 주문 금액, 주문번호

payment_sales = (
    order_sales
    .groupby('payment_method', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique')
    )
)

payment_sales['avg_order_value'] = (
    payment_sales['total_sales'] / payment_sales['order_count']
).round(0)

payment_sales

,payment_method,total_sales,order_count,avg_order_value
0,bank_transfer,58192000,74,786378.0
1,card,63541000,70,907729.0
2,kakao_pay,68326000,79,864886.0
3,naver_pay,65551000,77,851312.0


In [ ]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
# 그룹바이로 월별로 정리하고 카테고리 나눠지게, 금액을 카테고리 별로 항목별로 합쳐진 데이터로 요약표

monthly_category_sales = (
    order_sales
    .merge(products[['product_id', 'category']], on='product_id')
    .groupby(['order_month', 'category'], as_index=False)
    .agg(total_sales=('line_total', 'sum'))
)

monthly_category_sales


,order_month,category,total_sales
0,2025-09,도서,583000
1,2025-09,뷰티,1368000
2,2025-09,생활용품,1046000
3,2025-09,스포츠,269000
4,2025-09,식품,1186000
...,...,...,...
85,2026-09,생활용품,560000
86,2026-09,스포츠,800000
87,2026-09,식품,923000
88,2026-09,전자기기,1590000


In [38]:
# 5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
customer_sales.head(10)

#반복 구매 고객은 주문 횟수가 2회 이상인 고객, 단발 구매 고객은 1회인 고객으로 구분한다.
#고액 구매 여부는 고객별 평균 주문 금액의 전체 평균을 기준으로 판단한다.
#이를 통해 반복 구매 고객과 한 번에 높은 금액을 구매한 고객을 구분할 수 있을 것으로 생각한다.

,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


### 과제 6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.

온라인 쇼핑몰 데이터로 EDA를 진행하고 있습니다.

현재 사용할 수 있는 데이터는 다음과 같습니다.

- customers: 고객 ID, 나이, 성별, 도시, 가입일
- products: 상품 ID, 카테고리, 가격
- orders: 주문 ID, 고객 ID, 주문일, 결제수단, 주문 상태
- order_items: 주문 ID, 상품 ID, 수량, 단가, 주문 상품 금액

다음 질문을 현재 데이터로 분석할 수 있는 질문으로 바꿔 주세요.

"광고가 고객 구매 금액을 증가시켰는가?"

조건:
1. 현재 데이터에 없는 정보는 사용하지 마세요.
2. 수정한 질문에 필요한 컬럼과 계산 지표를 알려주세요.
3. 실제로 검증 가능한 질문을 제안해 주세요.
4. 분석 결과만으로 원인을 단정하지 마세요.

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


In [39]:
#Completed 상태 주문 확인
completed_sales = order_sales[
    order_sales['order_status'] == 'completed'
]

completed_total = completed_sales['line_total'].sum()

print('completed 원본 합계:', completed_total)
print('completed 주문 수:', completed_sales['order_id'].nunique())

completed 원본 합계: 148990000
completed 주문 수: 184


In [45]:
#카테고리별 검증
completed_category_sales = (
    completed_sales
    .merge(products[['product_id', 'category']], on='product_id')
    .groupby('category', as_index=False)
    .agg(
    total_quantity=('quantity', 'sum'),
    total_sales=('line_total', 'sum'))
)

print(completed_category_sales)
print('카테고리별 합계:', completed_category_sales['total_sales'].sum())

  category  total_quantity  total_sales
0       도서             149     16389000
1       뷰티             223     23383000
2     생활용품             272     23915000
3      스포츠             295     31743000
4       식품             133     16573000
5     전자기기             259     26400000
6       패션             111     10587000
카테고리별 합계: 148990000


In [47]:
#월별 합계 검증
completed_monthly_sales = (
    completed_sales
    .groupby('order_month', as_index=False)
    .agg(
    total_sales=('line_total', 'sum'),
    order_count=('order_id', 'nunique'))
)

print(completed_monthly_sales)
print('월별 합계:', completed_monthly_sales['total_sales'].sum())

   order_month  total_sales  order_count
0      2025-09      2650000            5
1      2025-10     16970000           19
2      2025-11     11021000           13
3      2025-12     21085000           23
4      2026-01     14272000           16
5      2026-02      4784000            7
6      2026-03     18323000           25
7      2026-04     10254000           15
8      2026-05     16831000           19
9      2026-06     12994000           17
10     2026-07     10652000           15
11     2026-08      5103000            5
12     2026-09      4051000            5
월별 합계: 148990000


In [49]:
customers['age_group'] = (customers['age'] // 10 * 10).astype(str) + '대'

customers['age_group'].value_counts().sort_index()

age_group
10대     5
20대    34
30대    33
40대    20
50대    27
60대    31
Name: count, dtype: int64

In [50]:
age_sales = (
    completed_sales
    .merge(customers[['customer_id', 'age_group']], on='customer_id')
    .groupby('age_group', as_index=False)
    .agg(total_sales=('line_total', 'sum'))
)

age_sales = (
    customers.groupby('age_group', as_index=False)
    .agg(customer_count=('customer_id', 'count'))
    .merge(age_sales, on='age_group', how='left')
)

age_sales['total_sales'] = age_sales['total_sales'].fillna(0)

age_sales['avg_sales_per_customer'] = (
    age_sales['total_sales'] / age_sales['customer_count']
).round(0)

age_sales

,age_group,customer_count,total_sales,avg_sales_per_customer
0,10대,5,4994000,998800.0
1,20대,34,35906000,1056059.0
2,30대,33,37992000,1151273.0
3,40대,20,18677000,933850.0
4,50대,27,19242000,712667.0
5,60대,31,32179000,1038032.0


# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

> 이 내용을 `chapter06.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 오현우
- GitHub ID: dhgusdn0312
- 작성일: 2026.10.21
- 최종 Notebook URL: https://github.com/dhgusdn0312/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

## 1. 내가 정한 EDA 질문
| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 카테고리별 완료 주문에서 금액이 높은 카테고리는 판매 수량도 많은가? | 2025-09-16~2026-09-15, completed 주문 | 카테고리별 주문 금액 합계, 판매 수량 합계 | 카테고리별 금액과 판매 수량 순위 비교, 금액 총합 검증 |
| 월별 완료 주문 금액이 높은 달은 주문 건수도 많은가? | 동일 기간, completed 주문 | 월별 주문 금액 합계, 고유 주문 건수 | 월별 금액과 주문 건수 비교, 금액 총합 검증 |
| 고객의 연령대에 따라 1인당 평균 완료 주문 금액에 차이가 있는가? | 동일 기간, completed 주문, 10세 단위 연령대 | 연령대별 완료 주문 총금액 ÷ 해당 연령대 전체 등록 고객 수 | 연령대별 평균 비교, 고객 수 및 금액 총합 검증 |

### 왜 이 질문을 선택했는가?
주문 금액이 높게 나타나는 것이 단순히 판매 수량이나 주문 건수가 많기 때문인지 궁금했습니다. 그래서 카테고리별 금액과 수량, 월별 금액과 주문 건수를 비교하는 질문을 선택했습니다.
또한 연령대별로 고객 수가 다르기 때문에 단순 총금액보다는 고객 1인당 평균 금액을 비교해 보고 싶었습니다.
현재 데이터에는 주문 상태, 날짜, 상품, 고객 나이, 수량, 금액 정보가 있어 세 가지 질문 모두 실제 데이터로 확인할 수 있습니다.

## 2. 핵심 집계 결과
- completed 원본 합계: 148,990,000원
- 카테고리별 합계: 148,990,000원
- 월별 합계: 148,990,000원
- 총합 일치 여부: 일치

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
Q1: 스포츠 카테고리가 완료 주문 금액 31,743,000원, 판매 수량 295개로 모두 가장 높았다. 반면 생활용품은 판매 수량이 272개로 전자기기(259개)보다 많았지만, 주문 금액은 전자기기가 더 높았다.

Q2: 2025년 12월은 완료 주문 금액이 21,085,000원으로 가장 높았지만, 주문 건수는 2026년 3월이 25건으로 가장 많았다.

Q3: 연령대별 고객 1인당 평균 완료 주문 금액은 30대가 약 115만 원으로 가장 높았고, 50대가 약 71만 원으로 가장 낮았다.

### 나의 해석과 판단
주문 금액이 높게 나타나는 것이 반드시 판매 수량이나 주문 건수가 가장 많기 때문만은 아니라고 생각한다. 상품 가격이나 주문당 구매 금액도 함께 살펴볼 필요가 있다.
연령대별 평균 금액에도 차이가 있었지만, 이 차이가 나타난 이유는 현재 결과만으로 판단하기 어렵다.

### 업무·분석적 의미
카테고리별 판매 수량과 금액을 함께 비교하면 어떤 상품군을 더 자세히 분석해야 할지 판단하는 데 도움이 된다.
또한 월별 주문 건수와 연령대별 구매 금액을 살펴보면 판매 현황과 고객 특성을 파악하고, 이후 상품 운영이나 고객 분석의 방향을 정하는 데 활용할 수 있다.

### 한계와 추가 확인 사항
카테고리별 금액 차이를 설명하려면 평균 판매 단가를 추가로 확인해야 한다.
월별 금액 차이는 평균 주문 금액과 판매 수량도 함께 분석해야 한다.
연령대별 차이는 실제 구매 고객 비율과 주문 횟수 등을 추가로 확인해야 한다.
2025년 9월과 2026년 9월은 일부 날짜만 포함되어 다른 월과 직접 비교하기 어렵다.
이번 금액은 line_total 합계로 계산했으며, 실제 정산 매출과 일치하는지는 확인하지 않았다.

## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: 스포츠 카테고리는 완료 주문 금액(31,743,000원)과 판매 수량(295개)이 모두 가장 높았다. 하지만 생활용품은 전자기기보다 판매 수량이 많았는데도 주문 금액은 더 낮았다.
- 가설: 카테고리별 주문 금액 차이는 판매 수량뿐 아니라 상품 단가의 차이와도 관련이 있을 수 있다.
- 추가 검증: 카테고리별 실제 판매 단가를 비교하고, 수량과 단가가 주문 금액에 어떻게 연결되는지 확인한다.

### 사례 2
- 관찰: 2025년 12월은 완료 주문 금액이 21,085,000원으로 가장 높았지만, 주문 건수는 2026년 3월이 25건으로 가장 많았다.
- 가설: 월별 주문 금액의 차이는 주문 건수뿐 아니라 주문 1건당 평균 금액의 차이와도 관련이 있을 수 있다.
- 추가 검증: 월별 평균 주문 금액을 계산하고, 판매 수량과 상품 구성에도 차이가 있는지 확인한다.

### 사례 3
- 관찰: 등록 고객 1인당 평균 완료 주문 금액은 30대가 약 115만 원으로 가장 높았고, 50대는 약 71만 원으로 가장 낮았다.
- 가설: 연령대별 구매 고객 비율이나 주문 횟수의 차이가 평균 금액에 영향을 주었을 가능성이 있다.
- 추가 검증: 연령대별 실제 구매 고객 비율, 1인당 주문 횟수, 주문 1건당 평균 금액을 비교한다

## 4. LLM 질문 확장
- Safe Context:
온라인 쇼핑몰의 고객, 상품, 주문, 주문 상세 데이터의 컬럼 구조만 제공했다. 고객 이름 등 실제 개인정보와 원본 데이터는 전달하지 않았다.

- Prompt:
온라인 쇼핑몰 데이터로 EDA를 진행하고 있습니다.
현재 데이터는 다음과 같습니다.

customers: customer_id, name, gender, age, city, signup_date
products: product_id, product_name, category, price
orders: order_id, customer_id, order_date, payment_method, order_status
order_items: order_item_id, order_id, product_id, quantity, unit_price, line_total

현재 분석에서는 completed 주문만 금액 분석 대상으로 사용합니다.
다음 조건에 맞춰 EDA 질문 5개를 제안해 주세요.
1.현재 데이터로 계산하고 검증할 수 있는 질문
2.각 질문에 필요한 데이터와 계산 지표
3.질문을 통해 확인하려는 목적
4.데이터에 없는 정보는 추측하지 말 것
5.상관관계를 원인으로 단정하지 말 것

- LLM 제안 요약:
1. 금액 데이터의 내부 일관성 검증
2. 카테고리·상품별 매출 집중도 분석
3. 월별 매출과 주문 건수·평균 주문 금액 비교
4. 결제수단별 주문 상태 및 금액 비교
5. 고객 속성별 구매 규모와 재구매율 비교

- 실제 사용 여부: 수정 후 사용
- 판단 이유: 다섯 질문 모두 현재 데이터로 분석할 수 있는 내용이지만, 일부는 추가 검증이 필요하다. 특히 Q5는 앞서 수행한 연령대별 구매 금액 분석과 연결할 수 있어 선택했다.
기존 분석에서는 30대의 고객 1인당 평균 완료 주문 금액이 가장 높았지만, 실제 구매한 고객의 비율 때문인지 구매 횟수나 금액 때문인지는 알 수 없었다.

- 사람이 수정한 내용:
LLM은 연령대, 성별, 도시별 구매 규모와 재구매율을 함께 분석하도록 제안했다. 하지만 분석 범위가 넓어 현재 질문과 직접 관련된 연령대별 구매 고객 비율로 좁혔다.
수정한 질문: 연령대별 등록 고객 중 실제 completed 주문을 한 고객의 비율은 어떻게 다른가?

![LLM EDA 질문 확장](images/step04_llm.png)

## 5. 다음 분석 우선순위
1. 연령대별 실제 구매 고객 비율 분석
2. 카테고리별 평균 판매 단가 분석
3. 월별 평균 주문 금액 분석

### 가장 먼저 확인하고 싶은 이유
연령대별 고객 1인당 평균 완료 주문 금액은 30대가 약 115만 원으로 가장 높았고, 50대는 약 71만 원으로 가장 낮았다.
이 차이가 실제 구매한 고객의 비율 때문인지, 구매 고객들의 주문 횟수나 금액 때문인지 궁금하다. 먼저 연령대별 구매 고객 비율을 확인하면 고객 특성을 좀 더 구체적으로 이해할 수 있을 것 같다.

### 현재 데이터만으로 단정할 수 없는 것
카테고리별 금액 차이가 판매 수량이나 단가 중 어느 요인과 더 관련이 있는지 추가 계산 없이 판단하기 어렵다.
월별 주문 금액 차이가 발생한 원인은 현재 데이터만으로 알 수 없다.
연령대별 구매 금액 차이가 나이 자체의 영향 때문이라고 단정할 수 없다.

## 최종 체크
- [/] 질문과 지표가 연결되어 있습니다.
- [/] 집계 총합을 검증했습니다.
- [/] 관찰과 가설을 구분했습니다.
- [/] 다음 검증 질문을 작성했습니다.
- [/] LLM 제안을 검증했습니다.
- [/] 최종 Notebook URL을 제출합니다.